In [1]:
import os
import sys
import pandas as pd
import json 
from pathlib import Path
import yaml
import shutil

# Add spcenarios to path
ROOT_DIR = os.path.abspath(os.path.join("../../brightpath"))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

from brightpath import bwconverter
from brightpath import utils
import re

16:27:00+0100 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


### Convert 1 LCI

In [3]:
fp="Zn_LCIs_BW_Final_v1.xlsx"

c = bwconverter.BrightwayConverter(
    filepath=fp,
    metadata="meta_sample_Zn.yaml",
    ecoinvent_version = "3.10",
    export_dir=r"C:\Users\istrateir\OneDrive - Universiteit Leiden\Research\brightpath\dev"
)

Extracted 6 worksheets in 0.05 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 16 strategies in 0.05 seconds


In [4]:
c.convert_to_simapro(database="ecoinvent")

Cannot find appropiate elementary flow for exchange: ('COD, Chemical Oxygen Demand', 'water', '')
Cannot find appropiate elementary flow for exchange: ('Suspended solids, unspecified', 'water', '')
Cannot find appropiate elementary flow for exchange: ('Sulfide', 'water', '')
Cannot find appropiate elementary flow for exchange: ('BOD5, Biological Oxygen Demand', 'water', '')
Cannot find appropiate elementary flow for exchange: ('Surfactant blend', 'water', 'surface water')
Cannot find appropiate elementary flow for exchange: ('COD, Chemical Oxygen Demand', 'water', '')
Cannot find appropiate elementary flow for exchange: ('COD, Chemical Oxygen Demand', 'water', '')
Cannot find appropiate elementary flow for exchange: ('Sulfide', 'water', '')
Cannot find appropiate elementary flow for exchange: ('Suspended solids, unspecified', 'water', '')
Cannot find appropiate elementary flow for exchange: ('Mineral oil', 'soil', 'agricultural')
Cannot find appropiate elementary flow for exchange: ('C

'Inventories export to: C:\\Users\\istrateir\\OneDrive - Universiteit Leiden\\Research\\brightpath\\dev\\simapro_ecoinvent_02-03-2026.csv'

### Convert all LCIs from Lai et al.

In [2]:
# Import data
lai_lci_path = Path(r"C:\Users\istrateir\OneDrive - Universiteit Leiden\Research\_Databases\Lai et al - LCI metal and minerals")
lai_lci_files = list(lai_lci_path.glob("*.xlsx"))

In [5]:
template_meta = "meta_sample.yaml"
converted_lci_path = Path(r"C:\Users\istrateir\OneDrive - Universiteit Leiden\Research\brightpath\dev\lci_metals_simapro")

unconverted_flows = []

for fp in lai_lci_files:
    metal = Path(fp).stem.split("_")[0]
    print("*********************")
    print(metal)
    print("*********************")
    
    with open(template_meta, "r") as f:
        meta_data = yaml.safe_load(f)

    # modify metadata based on filename
    meta_data["system description"]["name"] = f"{metal} production"
    meta_data["system description"]["category"] = f"Use/{metal}"
    meta_data["literature reference"]["documentation link"] = "https://doi.org/10.1016/j.resconrec.2025.108709"

    # Save temporary metadata file
    meta_file_path  = converted_lci_path / "metals_meta_samples" / f"meta_sample_{metal}.yaml"
    with open(meta_file_path , "w") as f:
        yaml.dump(meta_data, f)

    c = bwconverter.BrightwayConverter(
        filepath=fp,
        metadata=str(meta_file_path ),
        ecoinvent_version="3.10",
        export_dir=converted_lci_path
    )
    
    c.convert_to_simapro(database="ecoinvent")

    exported_file = os.path.join(converted_lci_path, "simapro_ecoinvent_02-03-2026.csv")
    new_file = os.path.join(converted_lci_path, f"LCI_{metal}.csv")
    shutil.move(exported_file, new_file)

*********************
Ag
*********************
Extracted 5 worksheets in 0.04 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 16 strategies in 0.02 seconds
Cannot find appropiate elementary flow for exchange: ('Water, salt, ocean', 'natural resource', 'in water')
All 185 exchanges 

### Tests (ignore)

In [7]:
filename = "ef31_simapro_elementary_flows.json"
filepath = filename

with open(filepath, "r", encoding="utf-8") as f:
    ef31_simapro_flows = json.load(f)

In [37]:
# Mapping of compartments between ecoinvent and Simapro
compartment_mapping = {
    "air": "Air",
    "water": "Water",
    "soil": "Soil",
    "natural resource": "Raw"
}

simapro_subcompartments = utils.get_simapro_subcompartments()

roman_numerals = ['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII', 'IX', 'X']
roman_numerals_patterns = [re.compile(rf'\b{rn}\b') for rn in roman_numerals]

# name in ecoinvent: name in simapro
special_name_cases = {
    "Particulate Matter, < 2.5 um": "Particulates, < 2.5 um",
    "Particulate Matter, > 2.5 um and < 10um": "Particulates, > 2.5 um, and < 10um",
    "Particulate Matter, > 10 um": "Particulates, > 2.5 um, and < 10um", # PM > 10 um don't exist in SimaPro nomenclature
    "Nitrogen": "Nitrogen, total",
    "Dioxins, measured as 2,3,7,8-tetrachlorodibenzo-p-dioxin": "Dioxin, 2,3,7,8 Tetrachlorodibenzo-p-",
    "Ammonium": "Ammonium, ion",
    "Water, well, in ground": "Water, well",
    'Hydrocarbons, chlorinated': "Hydrocarbons, unspecified",
    'Tin ion': "Tin",
    "Titanium ion": "Titanium"
}

special_regions = {
 'GCC': "IAI Area, Gulf Cooperation Council",
 'South America': "IAI Area, South America",
 'North America': "IAI Area, North America, without Quebec",
 'Europe': "Europe without Switzerland",
 'US-WECC': "WECC, US only",
 'Rest-of-Asia': "IAI Area, Asia, without China and GCC",
 'Russia and Rest-of-Europe': "IAI Area, Russia & RER w/o EU27 & EFTA",
 'Oceania': "UN-OCEANIA",
 'Africa': "IAI Area, Africa"
}

In [47]:
ef_without_equivalence = []

for fp in lai_lci_files:
    metal = Path(fp).stem.split("_")[0]
    print("*********************")
    print(metal)
    print("*********************")

    meta_file_path  = converted_lci_path / "metals_meta_samples" / f"meta_sample_{metal}.yaml"

    c = bwconverter.BrightwayConverter(
        filepath=fp,
        metadata=str(meta_file_path ),
        ecoinvent_version="3.10",
        export_dir=converted_lci_path
    )

    for ds in c.inventories:
        for exc in ds["exchanges"]:
            if exc["type"] == "biosphere":
                
                if ds["location"] in special_regions:
                    ef_location = special_regions[ds["location"]]
                else:
                    ef_location = ds["location"]

                # Get compartments
                ei_compartment = exc["categories"][0]
                simapro_compartment = compartment_mapping[ei_compartment]

                # Get subcompartments
                if len(exc["categories"]) > 1:
                    ei_subcompartment = exc["categories"][1]
                    simapro_subcompartment = simapro_subcompartments[ei_subcompartment]
                else:
                    ei_subcompartment = ""
                    simapro_subcompartment = "(unspecified)"
                
                print("**************************")
                print("BW2:", (exc["name"], ei_compartment, ei_subcompartment, exc["unit"], ds["location"]))

                # Determine search names
                if exc["name"] in special_name_cases:
                    # Handling the name for special cases:
                    search_ef_name = special_name_cases[exc["name"]]

                elif "non-fossil" in exc["name"]:
                    # SimaPro uses "biogenic" instead of "non-fossil"
                    search_ef_name = exc["name"].replace("non-fossil", "biogenic")
                
                elif re.search(r'\bion\b', exc["name"]):
                    # Flows with ion are named as "Copper ion" in ecoinvent and as "Copper, ion" in SimaPro
                    search_ef_name = exc["name"].replace(" ion", ", ion")

                elif any(p.search(exc["name"]) for p in roman_numerals_patterns):
                    # Flows with oxidation state like Silver I are named as Silver (I) in SimaPro
                    search_ef_name = re.sub(r'^(.*)\s+(I{1,3}|IV|V|VI{0,3}|IX|X)$', r'\1 (\2)', exc["name"])

                else:
                    search_ef_name = exc["name"]

                # Let's check if there is a regionalized flow
                for subcomp in [simapro_subcompartment, "(unspecified)"]:
                    simapro_ef_match = next(
                        (ff for ff in ef31_simapro_flows
                        if ff["name"] == f"{search_ef_name}, {ef_location}"
                        and ff["compartment"] == simapro_compartment
                        and ff["subcompartment"] == subcomp),
                        None
                    )
                    if simapro_ef_match:
                        break 
                
                # If no regionalized flows exist, try to match exact name
                if not simapro_ef_match:
                    for subcomp in [simapro_subcompartment, "(unspecified)"]:
                        simapro_ef_match = [
                            ff for ff in ef31_simapro_flows
                            if ff["name"] == search_ef_name
                            and ff["compartment"] == simapro_compartment
                            and ff["subcompartment"] == subcomp
                        ]
                        if simapro_ef_match:
                            break
                
                if not simapro_ef_match:
                    ef_without_equivalence.append((exc["name"], ei_compartment, ei_subcompartment, exc["unit"], ds["location"]))
    

*********************
Ag
*********************
Extracted 5 worksheets in 0.02 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 16 strategies in 0.02 seconds
**************************
BW2: ('Water, unspecified natural origin', 'natural resource', 'in ground', 'cubic meter', 'CN')
**

In [48]:
len(list(set(ef_without_equivalence)))

181

In [52]:
[i for i in ef31_simapro_flows if "Water" in i["name"] and "Air" in i["compartment"]]

[]

In [51]:
list(set(ef_without_equivalence))

[('Thorium-232',
  'air',
  'non-urban air or from high stacks',
  'kilo Becquerel',
  'ZA'),
 ('Tetrafluoromethane', 'air', '', 'kilogram', 'Rest-of-Asia'),
 ('Suspended solids, unspecified', 'water', '', 'kilogram', 'Oceania'),
 ('Cerium', 'natural resource', 'in ground', 'kilogram', 'AU'),
 ('Dimethenamide',
  'air',
  'non-urban air or from high stacks',
  'kilogram',
  'GLO'),
 ('Hydrogen', 'air', '', 'kilogram', 'US'),
 ('Sand, unspecified', 'natural resource', 'in ground', 'kilogram', 'CN'),
 ('Cerium', 'natural resource', 'in ground', 'kilogram', 'CN'),
 ('Heat, waste', 'air', '', 'megajoule', 'CN'),
 ('Nitrogen', 'air', '', 'kilogram', 'Europe'),
 ('Water, in air', 'natural resource', 'in air', 'cubic meter', 'GLO'),
 ('Heat, waste',
  'air',
  'non-urban air or from high stacks',
  'megajoule',
  'CN'),
 ('1,1,1-Trichloroethane',
  'air',
  'non-urban air or from high stacks',
  'kilogram',
  'ZA'),
 ('Thulium', 'natural resource', 'in ground', 'kilogram', 'AU'),
 ('1,2-Dichl

In [25]:
[i for i in list(set(ef_without_equivalence)) if i[0] == 'Water, unspecified natural origin']

[]

In [ ]:
ef_without_equivalence = []

for ds in c.inventories:
    for exc in ds["exchanges"]:
        if exc["type"] == "biosphere":
            
            ef_location = ds["location"]

            # Get compartments
            ei_compartment = exc["categories"][0]
            simapro_compartment = compartment_mapping[ei_compartment]

            # Get subcompartments
            if len(exc["categories"]) > 1:
                ei_subcompartment = exc["categories"][1]
                simapro_subcompartment = simapro_subcompartments[ei_subcompartment]
            else:
                ei_subcompartment = ""
                simapro_subcompartment = "(unspecified)"
            
            print("**************************")
            print("BW2:", (exc["name"], ei_compartment, ei_subcompartment, exc["unit"], ds["location"]))

                # Determine search names

            if re.search(r'\bion\b', exc["name"]):
                # Flows with ion are named as "Copper ion" in ecoinvent and as "Copper, ion" in SimaPro
                print(exc["name"])
                search_ef_name = exc["name"].replace(" ion", ", ion")

            elif any(p.search(exc["name"]) for p in roman_numerals_patterns):
                # Flows with oxidation state like Silver I are named as Silver (I) in SimaPro
                search_ef_name = re.sub(r'^(.*)\s+(I{1,3}|IV|V|VI{0,3}|IX|X)$', r'\1 (\2)', exc["name"])

            elif exc["name"] in special_name_cases:
                # Handling the name for special cases:
                search_ef_name = special_name_cases[exc["name"]]

            elif "non-fossil" in exc["name"]:
                # SimaPro uses "biogenic" instead of "non-fossil"
                search_ef_name = exc["name"].replace("non-fossil", "biogenic")

            else:
                search_ef_name = exc["name"]

            # Let's check if there is a regionalized flow
            for subcomp in [simapro_subcompartment, "(unspecified)"]:
                simapro_ef_match = next(
                    (ff for ff in ef31_simapro_flows
                    if ff["name"] == f"{search_ef_name}, {ef_location}"
                    and ff["compartment"] == simapro_compartment
                    and ff["subcompartment"] == subcomp),
                    None
                )
                if simapro_ef_match:
                    break 
            
            # If no regionalized flows exist, try to match exact name
            if not simapro_ef_match:
                for subcomp in [simapro_subcompartment, "(unspecified)"]:
                    simapro_ef_match = [
                        ff for ff in ef31_simapro_flows
                        if ff["name"] == search_ef_name
                        and ff["compartment"] == simapro_compartment
                        and ff["subcompartment"] == subcomp
                    ]
                    if simapro_ef_match:
                        break
            
            if not simapro_ef_match:
                ef_without_equivalence.append((exc["name"], ei_compartment, ei_subcompartment, exc["unit"], ds["location"]))
            print(simapro_ef_match)

**************************
BW2: ('Zinc', 'natural resource', 'in ground', 'kilogram', 'CN')
[{'compartment': 'Raw', 'subcompartment': '(unspecified)', 'name': 'Zinc', 'cas': '007440-66-6', 'unit': 'kg'}]
**************************
BW2: ('Water, unspecified natural origin', 'natural resource', 'in ground', 'cubic meter', 'CN')
{'compartment': 'Raw', 'subcompartment': '(unspecified)', 'name': 'Water, unspecified natural origin, CN', 'cas': '007732-18-5', 'unit': 'm3'}
**************************
BW2: ('Transformation, from unspecified', 'natural resource', 'land', 'square meter', 'CN')
[{'compartment': 'Raw', 'subcompartment': '(unspecified)', 'name': 'Transformation, from unspecified', 'cas': nan, 'unit': 'm2'}]
**************************
BW2: ('Transformation, to mineral extraction site', 'natural resource', 'land', 'square meter', 'CN')
[{'compartment': 'Raw', 'subcompartment': '(unspecified)', 'name': 'Transformation, to mineral extraction site', 'cas': nan, 'unit': 'm2'}]
***********

In [12]:
ef_without_equivalence

[('COD, Chemical Oxygen Demand', 'water', '', 'kilogram', 'CN'),
 ('Suspended solids, unspecified', 'water', '', 'kilogram', 'CN'),
 ('Sulfide', 'water', '', 'kilogram', 'CN'),
 ('BOD5, Biological Oxygen Demand', 'water', '', 'kilogram', 'CN'),
 ('Surfactant blend', 'water', 'surface water', 'kilogram', 'CN'),
 ('COD, Chemical Oxygen Demand', 'water', '', 'kilogram', 'CN'),
 ('COD, Chemical Oxygen Demand', 'water', '', 'kilogram', 'CN'),
 ('Sulfide', 'water', '', 'kilogram', 'CN'),
 ('Suspended solids, unspecified', 'water', '', 'kilogram', 'CN'),
 ('Mineral oil', 'soil', 'agricultural', 'kilogram', 'CN'),
 ('Tin ion', 'air', '', 'kilogram', 'CN'),
 ('COD, Chemical Oxygen Demand', 'water', '', 'kilogram', 'CN'),
 ('Sulfide', 'water', '', 'kilogram', 'CN'),
 ('Suspended solids, unspecified', 'water', '', 'kilogram', 'CN'),
 ('Water, salt, ocean', 'natural resource', 'in water', 'cubic meter', 'SE')]

In [130]:
[f for f in ef31_simapro_flows if "Water, well" in f["name"]]

[{'compartment': 'Raw',
  'subcompartment': '(unspecified)',
  'name': 'Water, well',
  'cas': '007732-18-5',
  'unit': 'm3'},
 {'compartment': 'Raw',
  'subcompartment': '(unspecified)',
  'name': 'Water, well, AD',
  'cas': '007732-18-5',
  'unit': 'm3'},
 {'compartment': 'Raw',
  'subcompartment': '(unspecified)',
  'name': 'Water, well, AE',
  'cas': '007732-18-5',
  'unit': 'm3'},
 {'compartment': 'Raw',
  'subcompartment': '(unspecified)',
  'name': 'Water, well, AF',
  'cas': '007732-18-5',
  'unit': 'm3'},
 {'compartment': 'Raw',
  'subcompartment': '(unspecified)',
  'name': 'Water, well, AG',
  'cas': '007732-18-5',
  'unit': 'm3'},
 {'compartment': 'Raw',
  'subcompartment': '(unspecified)',
  'name': 'Water, well, AI',
  'cas': '007732-18-5',
  'unit': 'm3'},
 {'compartment': 'Raw',
  'subcompartment': '(unspecified)',
  'name': 'Water, well, AL',
  'cas': '007732-18-5',
  'unit': 'm3'},
 {'compartment': 'Raw',
  'subcompartment': '(unspecified)',
  'name': 'Water, well, AM

In [ ]:

            
            simapro_flows_list = ef31_bw_simapro[(exc["name"], exc["categories"][0], sub_compartment, exc["unit"])]
            simapro_flow_match = None

            # If there is no SimaPro equivalence, use ecoinvent naming
        #    if len(simapro_exc_list) == 0:
        #        simapro_flow_match = (exc["name"], exc["categories"][0], sub_compartment, exc["unit"])
        #        ef_without_equivalence.append(simapro_flow_match)
            
            # If there is an exact match, use that one
            if len(simapro_flows_list) == 1:
                simapro_flow_match = simapro_flows_list
            
            # If there is more than one match
            if len(simapro_flows_list) > 1:
                
                # Try to find the regionalized flow
                for altv in simapro_flows_list:
                    if altv[0] == exc["name"] + ", " + ds["location"]:
                        simapro_flow_match = altv
                        break

                # If there is no regionalized flow, try to find the exact match
                if simapro_flow_match is None:
                    simapro_flow_match = [ff for ff in simapro_flows_list if ff[0] == exc["name"]]
                    if len(simapro_flow_match) == 0:
                        simapro_flow_match = None

            if simapro_flow_match is None:
                ef_without_equivalence.append((exc["name"], exc["categories"][0], sub_compartment))

            print(simapro_flow_match)
            print("**************************")
print("******************************")
print("Elementary flows used in ecoinvent without an equivalence in SimaPro:")
print("******************************")
print(ef_without_equivalence)

**************************
BW2: ('Zinc', 'natural resource', 'in ground', 'kilogram', 'CN')
[['Zinc', 'Raw', '(unspecified)', 'kg']]
**************************
**************************
BW2: ('Water, unspecified natural origin', 'natural resource', 'in ground', 'cubic meter', 'CN')
['Water, unspecified natural origin, CN', 'Raw', '(unspecified)', 'm3']
**************************
**************************
BW2: ('Transformation, from unspecified', 'natural resource', 'land', 'square meter', 'CN')
[['Transformation, from unspecified', 'Raw', '(unspecified)', 'm2']]
**************************
**************************
BW2: ('Transformation, to mineral extraction site', 'natural resource', 'land', 'square meter', 'CN')
[['Transformation, to mineral extraction site', 'Raw', '(unspecified)', 'm2']]
**************************
**************************
BW2: ('Transformation, from unspecified', 'natural resource', 'land', 'square meter', 'CN')
[['Transformation, from unspecified', 'Raw', '(u

In [76]:
ef_without_equivalence

[('COD, Chemical Oxygen Demand', 'water', ''),
 ('Suspended solids, unspecified', 'water', ''),
 ('Sulfide', 'water', ''),
 ('Chromium III', 'water', ''),
 ('Nickel II', 'water', ''),
 ('BOD5, Biological Oxygen Demand', 'water', ''),
 ('Surfactant blend', 'water', 'surface water'),
 ('Carbon dioxide, non-fossil', 'air', ''),
 ('Carbon monoxide, non-fossil', 'air', ''),
 ('COD, Chemical Oxygen Demand', 'water', ''),
 ('COD, Chemical Oxygen Demand', 'water', ''),
 ('Ammonium', 'water', ''),
 ('Chromium III', 'water', ''),
 ('Nickel II', 'water', ''),
 ('Sulfide', 'water', ''),
 ('Suspended solids, unspecified', 'water', ''),
 ('Chromium III', 'soil', ''),
 ('Nickel II', 'soil', ''),
 ('Mineral oil', 'soil', 'agricultural'),
 ('Chromium III', 'air', ''),
 ('Tin ion', 'air', ''),
 ('COD, Chemical Oxygen Demand', 'water', ''),
 ('Chromium III', 'water', ''),
 ('Nickel II', 'water', ''),
 ('Sulfide', 'water', ''),
 ('Suspended solids, unspecified', 'water', ''),
 ('Chromium III', 'soil', '')

In [ ]:
for ds in bw_lci:
    print("****************")
    print(ds["name"], ds["location"])
    print("****************")
    for exc in ds["exchanges"]:
        if exc["type"] == "biosphere":
            
            if len(exc["categories"]) > 1:
                exc_compartment = exc["categories"][0]
                sub_compartment = c.simapro_subcompartment[exc["categories"][1]]
            else:
                exc_compartment = exc["categories"]
                sub_compartment = ""

            simapro_exc_list = [se for se in ef31_bw_simapro
                        if se[0] == exc["name"]
                        and se[1] == exc_compartment
                        and se[3] == exc["unit"]
                        and se[4] == exc["type"]]
            
            simapro_biosphere_flow = None

            if len(simapro_exc_list) == 0:
                simapro_biosphere_flow = exc["name"]
        
            # Check if exchange is regionalized in the simapro-biosphere mapping file
            # Some additional natural resource flows have to be regionalized
            if len(simapro_exc_list) == 1:
            
                for other_flows in ADDITIONAL_REGIONALIZED_FLOWS:
                    if other_flows in simapro_exc_list[0][1]:
                        simapro_biosphere_flow = simapro_exc_list[0][2] + ", " + ds["location"]
                        break

                if simapro_biosphere_flow is None:
                    simapro_biosphere_flow = simapro_exc_list[0]
        
            if len(simapro_exc_list) > 1:
                exc_candidate = [se for se in simapro_exc_list 
                        if se[2] == exc["name"]
                        and ds["location"] in se[1]
                        and sub_compartment in se[1]]
                if len(exc_candidate) == 1:
                    simapro_biosphere_flow = exc_candidate[0][2] + ", " + ds["location"]
                else:
                    raise ValueError("Cannot find appropiate biosphere flow for exchange:", exc["name"])

            # Name of the flow in SimaPro format: ecoinvent exchange name + country
            print("BW:", exc["name"], "||", "Simapro:", simapro_biosphere_flow, "//Subcompartment:", sub_compartment)


****************
Zinc (Zn) global market GLO
****************
****************
Zinc (Zn) global market (Zn content) GLO
****************
****************
[M+C] Zinc (Zn) ore mining and concentration (Jiangxi) CN
****************
BW: Zinc || Simapro: Zinc //Subcompartment: in ground
BW: Water, unspecified natural origin || Simapro: Water, unspecified natural origin, CN //Subcompartment: in ground
BW: Transformation, from unspecified || Simapro: Transformation, from unspecified, CN //Subcompartment: land
BW: Transformation, to mineral extraction site || Simapro: Transformation, to mineral extraction site, CN //Subcompartment: land
****************
[P+R] Refined Zn production - Hydrometallurgy CN
****************
BW: Transformation, from unspecified || Simapro: Transformation, from unspecified, CN //Subcompartment: land
BW: Transformation, to mineral extraction site || Simapro: Transformation, to mineral extraction site, CN //Subcompartment: land
BW: Water, unspecified natural origin || S

In [13]:
import pandas as pd

fp="/Users/romain/OneDrive/Documents/Power2X/simapro_uvek_16-08-2023.csv"
exp_dir = "/Users/romain/OneDrive/Documents/Power2X/formatted_simapro_lci.xlsx"

# Function to parse individual lines
def parse_line_final_corrected(line, category):
    """Parses a line of data into various columns, corrected for resource flows, float precision, and amount extraction."""
    parts = line.split(";")
    name = parts[0] if len(parts) > 0 else None
    location = name.rsplit("/", 1)[1].split()[0] if "/" in name else "RER"
    category_name = category if name else None
    subcategory = "high population density" if "Emissions" in category else "-"
    infrastructure_process = "1" if "Products" in category else "0"
    unit = parts[1] if len(parts) > 1 else None
    product_name = None
    
    # Adjusting the position for resource flow amounts
    if "Resources" in category:
        amount = "{:.2e}".format(float(parts[3]))
    elif "Emissions" in category and len(parts) > 3:
        amount = "{:.2e}".format(float(parts[3]))
    elif len(parts) > 2:
        amount = "{:.2e}".format(float(parts[2]))
    else:
        amount = None
        
    uncertainty_type = 1 if name else None
    standard_deviation = None
    general_comment = parts[-1] if len(parts) > 3 else None
    return name, location, category_name, subcategory, infrastructure_process, unit, product_name, amount, uncertainty_type, standard_deviation, general_comment

# Function to parse the entire dataset into a DataFrame
def parse_dataset_to_df_updated(dataset):
    """Parses a dataset into a DataFrame with the specified format, adding headers."""
    categories = {
        "Products": "Output",
        "Materials/fuels": "Material inputs",
        "Electricity/heat": "Energy inputs",
        "Resources": "Resources",
        "Emissions to air": "Emissions to air",
        "Emissions to water": "Emissions to water",
        "Emissions to soil": "Emissions to soil",
        "Waste to treatment": "Waste treatment",
        "Final waste flows": "Final waste flows",
        "Non material emission": "Non material emission",
        "Social issues": "Social issues",
        "Economic issues": "Economic issues",
        "End": None
    }
    
    parsed_data = []
    current_category = None
    
    for line in dataset:
        line = line.strip()
        if line in categories:
            current_category = line
            if categories[current_category] is not None:
                parsed_data.append((categories[current_category], None, None, None, None, None, None, None, None, None, None))
                categories[current_category] = None
        elif current_category and line:
            parsed_line = parse_line_final_corrected(line, current_category)
            parsed_data.append(parsed_line)
        elif not line and parsed_data:
            parsed_data.append((None, None, None, None, None, None, None, None, None, None, None))
    
    df = pd.DataFrame(parsed_data, columns=["Name", "Location", "Category", "SubCategory", "InfrastructureProcess", "Unit", "Product", "Amount", "UncertaintyType", "StandardDeviation95%", "GeneralComment"])
    return df

# Splitting the uploaded file into individual datasets
with open(fp, "r") as file:
    lines = file.readlines()

datasets = []
current_dataset = []
for line in lines:
    if "Process:" in line:
        if current_dataset:
            datasets.append(current_dataset)
        current_dataset = [line]
    else:
        current_dataset.append(line)
datasets.append(current_dataset)

# Parsing and saving the datasets into an Excel file
all_tables_final_corrected = pd.DataFrame()

for i, dataset in enumerate(datasets):
    df = parse_dataset_to_df_updated(dataset)
    df.index = df.index + len(all_tables_final_corrected) + 2
    all_tables_final_corrected = pd.concat([all_tables_final_corrected, df])
    all_tables_final_corrected = pd.concat([all_tables_final_corrected, pd.DataFrame([["","","","","","","","","","",""]], columns=["Name", "Location", "Category", "SubCategory", "InfrastructureProcess", "Unit", "Product", "Amount", "UncertaintyType", "StandardDeviation95%", "GeneralComment"])])

all_tables_final_corrected.to_excel(exp_dir, index=False)
